# Data Workflow Project: Titanic Dataset Analysis

**Author:** [Your Name]

**Dataset:** Titanic - Machine Learning from Disaster ([Kaggle Link](https://www.kaggle.com/c/titanic))

**Project Description:** This project demonstrates a complete, reproducible data workflow using the Titanic passenger dataset. The workflow includes data ingestion, cleaning, exploratory analysis, and visualization to understand factors that influenced passenger survival rates during the Titanic disaster.

---
## 1. Setup: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

---
## 2. Data Ingestion

In [ ]:
# Load the Titanic dataset
df = pd.read_csv('titanic.csv')

# Display first few rows
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# Check data types and missing values
df.info()

---
## 3. Data Cleaning

The dataset has missing values in the `Age`, `Cabin`, and `Embarked` columns. We will create cleaning functions to handle these issues.

In [ ]:
def fill_missing_age(dataframe, group_cols=['Pclass', 'Sex']):
    """
    Fill missing Age values using the median age grouped by passenger class and sex.
    
    This approach preserves the relationship between age and other demographic factors
    rather than using a single global median, which would ignore important correlations.
    
    Parameters:
    -----------
    dataframe : pandas.DataFrame
        The input dataframe containing an 'Age' column
    group_cols : list
        Columns to group by when calculating median age
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with missing Age values filled
    """
    df_copy = dataframe.copy()
    
    # Calculate median age for each group
    median_ages = df_copy.groupby(group_cols)['Age'].transform('median')
    
    # Fill missing ages with group median
    df_copy['Age'] = df_copy['Age'].fillna(median_ages)
    
    # If any still missing (edge case), fill with overall median
    df_copy['Age'] = df_copy['Age'].fillna(df_copy['Age'].median())
    
    print(f"Filled {dataframe['Age'].isna().sum()} missing Age values")
    return df_copy

In [ ]:
def clean_categorical_columns(dataframe):
    """
    Clean categorical columns by filling missing values and standardizing formats.
    
    - Fills missing 'Embarked' values with the mode (most common port)
    - Creates a 'HasCabin' binary feature from the Cabin column
    - Drops the original Cabin column (too many missing values to impute meaningfully)
    
    Parameters:
    -----------
    dataframe : pandas.DataFrame
        The input dataframe
    
    Returns:
    --------
    pandas.DataFrame
        Cleaned dataframe with processed categorical columns
    """
    df_copy = dataframe.copy()
    
    # Fill missing Embarked with mode
    embarked_mode = df_copy['Embarked'].mode()[0]
    missing_embarked = df_copy['Embarked'].isna().sum()
    df_copy['Embarked'] = df_copy['Embarked'].fillna(embarked_mode)
    print(f"Filled {missing_embarked} missing Embarked values with '{embarked_mode}'")
    
    # Create HasCabin feature (1 if cabin is known, 0 otherwise)
    df_copy['HasCabin'] = df_copy['Cabin'].notna().astype(int)
    print(f"Created 'HasCabin' feature from Cabin column")
    
    # Drop original Cabin column
    df_copy = df_copy.drop('Cabin', axis=1)
    print("Dropped original 'Cabin' column (77% missing)")
    
    return df_copy

In [ ]:
# Apply cleaning functions
print("=" * 50)
print("APPLYING DATA CLEANING FUNCTIONS")
print("=" * 50)

df_clean = fill_missing_age(df)
print()
df_clean = clean_categorical_columns(df_clean)

print("\n" + "=" * 50)
print("CLEANING COMPLETE")
print("=" * 50)

In [ ]:
# Verify no missing values remain in key columns
print("Missing values after cleaning:")
print(df_clean.isnull().sum())

---
## 4. Exploratory Data Analysis (EDA)

In [ ]:
def perform_eda(dataframe, target_col='Survived'):
    """
    Perform exploratory data analysis on the dataset.
    
    Generates summary statistics, survival rates by key categories,
    and correlation analysis for numeric features.
    
    Parameters:
    -----------
    dataframe : pandas.DataFrame
        The cleaned dataframe to analyze
    target_col : str
        The target variable column name
    
    Returns:
    --------
    dict
        Dictionary containing EDA results
    """
    results = {}
    
    # Basic statistics
    print("=" * 60)
    print("SUMMARY STATISTICS")
    print("=" * 60)
    results['summary_stats'] = dataframe.describe()
    print(results['summary_stats'])
    
    # Overall survival rate
    print("\n" + "=" * 60)
    print("OVERALL SURVIVAL RATE")
    print("=" * 60)
    survival_rate = dataframe[target_col].mean() * 100
    print(f"Overall survival rate: {survival_rate:.1f}%")
    results['overall_survival'] = survival_rate
    
    # Survival by passenger class
    print("\n" + "=" * 60)
    print("SURVIVAL RATE BY PASSENGER CLASS")
    print("=" * 60)
    survival_by_class = dataframe.groupby('Pclass')[target_col].agg(['mean', 'count'])
    survival_by_class.columns = ['Survival Rate', 'Count']
    survival_by_class['Survival Rate'] = (survival_by_class['Survival Rate'] * 100).round(1)
    print(survival_by_class)
    results['survival_by_class'] = survival_by_class
    
    # Survival by sex
    print("\n" + "=" * 60)
    print("SURVIVAL RATE BY SEX")
    print("=" * 60)
    survival_by_sex = dataframe.groupby('Sex')[target_col].agg(['mean', 'count'])
    survival_by_sex.columns = ['Survival Rate', 'Count']
    survival_by_sex['Survival Rate'] = (survival_by_sex['Survival Rate'] * 100).round(1)
    print(survival_by_sex)
    results['survival_by_sex'] = survival_by_sex
    
    # Correlation matrix for numeric columns
    print("\n" + "=" * 60)
    print("CORRELATION WITH SURVIVAL")
    print("=" * 60)
    numeric_cols = dataframe.select_dtypes(include=[np.number]).columns
    correlations = dataframe[numeric_cols].corr()[target_col].sort_values(ascending=False)
    print(correlations)
    results['correlations'] = correlations
    
    return results

In [ ]:
# Run EDA
eda_results = perform_eda(df_clean)